In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.features import geometry_mask, geometry_window
from rasterio.windows import Window
from shapely.geometry import mapping
from shapely import make_valid
from shapely.validation import explain_validity


# ============================================================
# 1. Configuration
# ============================================================

BASE = Path("../data")

SMOD = BASE / "smod_0514.shp"
PANEL = Path("../data/study_grid_cells.csv")

BUILDINGS = Path("../data/raw/open_buildings")

OUT_BASE = Path(
    "../outputs/building"
)

YEARS = list(range(2016, 2024))

COVERAGE_MIN = 0.90
BLOCK = 512
RETRIES = 4

CONFIG = {
    "Abidjan_IvoryCoast":
        ("Abidjan", "Abidjan", 32630),
    "AddisAbaba_Ethiopia":
        ("AddisAbaba", "Addis Ababa", 32637),
    "Alexandria_Egypt":
        ("Alexandria", "Alexandria", 32636),
    "Bamako_Mali":
        ("Bamako", "Bamako", 32629),
    "Bangalore_India":
        ("Bangalore", "Bangalore", 32643),
    "Bangkok_Thailand":
        ("Bangkok", "Bangkok", 32647),
    "BeloHorizonte_Brazil":
        ("BeloHorizonte", "Belo Horizonte", 32723),
    "Bogota_Colombia":
        ("Bogota", "Bogota", 32618),
    "BuenosAires_Argentina":
        ("Buenos", "Buenos Aires", 32721),
    "Cairo_Egypt":
        ("Cairo", "Cairo", 32636),
    "Chennai_India":
        ("Chennai", "Chennai", 32644),
    "Colombo_SriLanka":
        ("Colombo", "Colombo", 32644),
    "DaresSalaam_Tanzania":
        ("DaresSalaam", "Dar es Salaam", 32737),
    "Delhi_India":
        ("Delhi", "Delhi", 32643),
    "Dhaka_Bangladesh":
        ("Dhaka", "Dhaka", 32645),
    "Guayaquil_Ecuador":
        ("Guayaquil", "Guayaquil", 32717),
    "HoChiMinh_Vietnam":
        ("Ho Chi Minh City", "Ho Chi Minh City", 32648),
    "Hyderabad_India":
        ("Hyderabad", "Hyderabad", 32644),
    "Jakarta_Indonesia":
        ("Jakarta", "Jakarta", 32748),
    "Johannesburg_SouthAfrica":
        ("Johannesburg", "Johannesburg", 32735),
    "Karachi_Pakistan":
        ("Karachi", "Karachi", 32642),
    "Kinshasa_DRC":
        ("Kinshasa", "Kinshasa", 32734),
    "KualaLumpur_Malaysia":
        ("Kuala Lumpur", "Kuala Lumpur", 32647),
    "Lagos_Nigeria":
        ("Lagos", "Lagos", 32631),
    "Lahore_Pakistan":
        ("Lahore", "Lahore", 32642),
    "Lima_Peru":
        ("Lima", "Lima", 32718),
    "Manaus_Brazil":
        ("Manaus", "Manaus", 32720),
    "Manila_Philippines":
        ("Manila", "Manila", 32651),
    "MexicoCity_Mexico":
        ("Mexico", "Mexico City", 32614),
    "Mumbai_India":
        ("Mumbai", "Mumbai", 32643),
    "Nairobi_Kenya":
        ("Nairobi", "Nairobi", 32737),
    "PhnomPenh_Cambodia":
        ("PhnomPenh", "Phnom Penh", 32648),
    "Santiago_Chile":
        ("Santiago", "Santiago", 32719),
    "SantoDomingo_DominicanRepublic":
        ("Santo Domingo", "Santo Domingo", 32619),
    "SaoPaulo_Brazil":
        ("Sao Paulo", "Sao Paulo", 32723),
}

RAW_SINGLE = {
    "Chennai_India":
        ("SA07_Chennai", "Chennai"),
    "HoChiMinh_Vietnam":
        ("SEA05_Ho Chi Minh City_Vietnam", "Ho_Chi_Minh_City"),
    "Kinshasa_DRC":
        ("AF08_Kinshasa", "Kinshasa"),
    "KualaLumpur_Malaysia":
        ("SEA06_Kuala_Lumpur", "KualaLumpur"),
    "Lahore_Pakistan":
        ("SA04_Lahore_Pakistan", "Lahore"),
    "Lima_Peru":
        ("LA02_Lima_Peru", "Lima"),
    "PhnomPenh_Cambodia":
        ("SEA07_Phnom_Penh", "Phnom_Penh"),
    "SantoDomingo_DominicanRepublic":
        ("LA09_Santo_Domingo", "Santo_Domingo"),
}

METRICS = [
    "n_bld_pixels",
    "bld_footprint_m2",
    "bld_mean_h_m",
    "bld_median_h_m",
    "bld_max_h_m",
    "bld_p90_h_m",
    "bld_std_h_m",
    "bld_var_h_m2",
    "bld_IQR_m",
    "bld_CV",
    "bld_low_share",
    "bld_mid_share",
    "bld_highrise_share",
    "gross_floor_area_m2",
    "bld_FAR",
    "bld_volume_m3",
    "bld_volume_density_m",
    "bld_coverage",
]

KEY = ["city", "cell_id", "year"]


# ============================================================
# 2. Common functions
# ============================================================

def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def stamp(path):
    s = Path(path).stat()
    return {
        "bytes": s.st_size,
        "mtime_ns": s.st_mtime_ns,
    }


def source_for(city, year):
    if city in RAW_SINGLE:
        folder, prefix = RAW_SINGLE[city]
        return (
            BUILDINGS / folder / f"{prefix}_{year}.tif",
            "open_buildings_raster",
        )

    prefix = CONFIG[city][0]

    return (
        BUILDINGS / prefix / f"{prefix}_{year}.tif",
        "open_buildings_raster",
    )


def read_retry(src, win):
    last = None

    for attempt in range(RETRIES):
        try:
            if attempt == 0:
                return src.read(
                    [1, 2],
                    window=win,
                    masked=True,
                )

            with rasterio.open(src.name) as reopened:
                return reopened.read(
                    [1, 2],
                    window=win,
                    masked=True,
                )

        except (rasterio.errors.RasterioIOError, OSError) as exc:
            last = exc

            if attempt + 1 < RETRIES:
                time.sleep(2 * (attempt + 1))

    raise RuntimeError(
        f"Read failed; must not be treated as no buildings: {src.name} {win}"
    ) from last


# ============================================================
# 3. The 18 metrics within the observed area
# ============================================================

def observed_metrics(values, n_valid, pixel_area):
    result = dict.fromkeys(METRICS, np.nan)

    if n_valid == 0:
        return result

    n = len(values)
    total_h = float(np.sum(values, dtype="float64"))

    result.update(
        n_bld_pixels=n,
        bld_footprint_m2=n * pixel_area,
        gross_floor_area_m2=total_h / 3.0 * pixel_area,
        bld_FAR=total_h / 3.0 / n_valid,
        bld_volume_m3=total_h * pixel_area,
        bld_volume_density_m=total_h / n_valid,
        bld_coverage=n / n_valid,
    )

    if n == 0:
        return result

    mean = float(np.mean(values))
    std = float(np.std(values, ddof=0))

    result.update(
        bld_mean_h_m=mean,
        bld_median_h_m=float(np.median(values)),
        bld_max_h_m=float(np.max(values)),
        bld_p90_h_m=float(
            np.percentile(values, 90, method="linear")
        ),
        bld_std_h_m=std,
        bld_var_h_m2=float(np.var(values, ddof=0)),
        bld_IQR_m=float(
            np.percentile(values, 75, method="linear")
            - np.percentile(values, 25, method="linear")
        ),
        bld_CV=std / mean,
        bld_low_share=float(
            np.mean((values >= 2) & (values < 10))
        ),
        bld_mid_share=float(
            np.mean((values >= 10) & (values < 25))
        ),
        bld_highrise_share=float(
            np.mean(values >= 25)
        ),
    )

    return result


# ============================================================
# 4. Computation for a single grid cell and a single year
# ============================================================

def calculate_cell(src, target, cell_area_m2, cell_area_deg2):
    counts = {
        "study_pixels": 0,
        "valid_pixels": 0,
        "missing_inside_pixels": 0,
        "outside_extent_pixels": 0,
        "minus9999_pixels": 0,
    }

    chunks = []
    transform = src.transform

    pixel_area = (
        cell_area_m2
        * abs(transform.a * transform.e)
        / cell_area_deg2
    )

    if not target.is_empty and target.area > 0:
        shapes = [mapping(target)]

        full = geometry_window(
            src,
            shapes,
            boundless=True,
        )

        x0 = int(full.col_off)
        y0 = int(full.row_off)
        x1 = x0 + int(full.width)
        y1 = y0 + int(full.height)

        for top in range(y0, y1, BLOCK):
            for left in range(x0, x1, BLOCK):
                height = min(BLOCK, y1 - top)
                width = min(BLOCK, x1 - left)

                win = Window(left, top, width, height)

                inside = geometry_mask(
                    shapes,
                    out_shape=(height, width),
                    transform=src.window_transform(win),
                    invert=True,
                    all_touched=False,
                )

                n = int(inside.sum())

                if n == 0:
                    continue

                counts["study_pixels"] += n

                c0 = max(left, 0)
                r0 = max(top, 0)
                c1 = min(left + width, src.width)
                r1 = min(top + height, src.height)

                if c1 <= c0 or r1 <= r0:
                    counts["outside_extent_pixels"] += n
                    continue

                selected = inside[
                    r0 - top:r1 - top,
                    c0 - left:c1 - left,
                ]

                counts["outside_extent_pixels"] += (
                    n - int(selected.sum())
                )

                if not selected.any():
                    continue

                arr = read_retry(
                    src,
                    Window(c0, r0, c1 - c0, r1 - r0),
                )

                values = np.asarray(arr.data)

                valid = ~(
                    np.ma.getmaskarray(arr)
                    | ~np.isfinite(values)
                    | (values == -9999)
                ).any(axis=0)

                counts["valid_pixels"] += int(
                    (selected & valid).sum()
                )

                counts["missing_inside_pixels"] += int(
                    (selected & ~valid).sum()
                )

                counts["minus9999_pixels"] += int(
                    (
                        selected
                        & (values == -9999).any(axis=0)
                    ).sum()
                )

                building = (
                    selected
                    & valid
                    & (values[0] >= 0.5)
                    & (values[1] >= 2.0)
                )

                if building.any():
                    chunks.append(
                        values[1][building].astype("float32")
                    )

    require(
        counts["study_pixels"]
        == (
            counts["valid_pixels"]
            + counts["missing_inside_pixels"]
            + counts["outside_extent_pixels"]
        ),
        "Covered-pixel counts are not conserved.",
    )

    heights = (
        np.concatenate(chunks)
        if chunks
        else np.empty(0, dtype="float32")
    )

    n = counts["study_pixels"]
    n_valid = counts["valid_pixels"]

    coverage = n_valid / n if n else np.nan

    passed = bool(
        n > 0 and coverage >= COVERAGE_MIN
    )

    complete = bool(
        n > 0 and n_valid == n
    )

    stats = observed_metrics(
        heights,
        n_valid,
        pixel_area,
    )

    if complete:
        status = "complete"
    elif passed:
        status = "near_complete"
    else:
        status = "below_coverage_threshold"

    if n == 0:
        status = "no_study_pixels"
    elif n_valid == 0:
        status = "no_valid_observations"

    counts.update(
        coverage_fraction=coverage,
        coverage_status=status,
        coverage_pass=passed,
        coverage_complete=complete,
        height_available=bool(len(heights) > 0),

        eligible_density_model=passed,
        eligible_height_model=bool(
            passed and len(heights) > 0
        ),

        observed_building_pixels=len(heights),
        pixel_area_m2_local=pixel_area,
        pixelized_study_area_m2=n * pixel_area,
        observed_area_m2=n_valid * pixel_area,
    )

    for metric, value in stats.items():
        counts[metric + "_observed"] = value

        counts[metric] = value if passed else np.nan

    return counts


# ============================================================
# 5. Main program
# ============================================================

def run_building_smod():
    sources = []
    tasks = []
    grid_rows = []

    def register(path, role, **extra):
        require(
            path.is_file(),
            f"Missing input: {path}",
        )

        sources.append({
            "path": str(path),
            "role": role,
            **stamp(path),
            **extra,
        })

    register(PANEL, "study_grid_panel")

    for ext in [".shp", ".shx", ".dbf", ".prj"]:
        register(
            SMOD.with_suffix(ext),
            "smod_boundary",
        )

    pts = pd.read_csv(PANEL)[
        ["city", "cell_id", "lat", "lon"]
    ].drop_duplicates()

    require(
        len(pts) == 652
        and set(pts["city"]) == set(CONFIG)
        and not pts.duplicated(
            ["city", "cell_id"]
        ).any(),
        "Abnormal list or coordinates of the 652 grid cells.",
    )

    # --------------------------------------------------------
    # 5.1 Read SMOD and repair geometry topology
    # --------------------------------------------------------

    boundaries = gpd.read_file(SMOD)

    require(
        boundaries.crs is not None,
        "SMOD has no CRS.",
    )

    boundaries = boundaries.to_crs(4326)

    geometry_repairs = []

    for idx, row in boundaries.iterrows():
        geom = row.geometry
        city_name = str(row["name"])

        require(
            geom is not None and not geom.is_empty,
            f"SMOD geometry of {city_name} is empty.",
        )

        if geom.is_valid:
            continue

        reason = explain_validity(geom)
        fixed = make_valid(geom)

        require(
            fixed.geom_type in ["Polygon", "MultiPolygon"]
            and fixed.is_valid
            and not fixed.is_empty,
            f"{city_name} is not a valid polygon after repair; check separately.",
        )

        area_difference = abs(fixed.area - geom.area)

        require(
            area_difference <= max(
                1e-12,
                abs(geom.area) * 1e-9,
            )
            and np.allclose(
                fixed.bounds,
                geom.bounds,
                atol=1e-10,
                rtol=0,
            ),
            f"Repair changed the area or extent of {city_name}; check separately.",
        )

        boundaries.at[idx, "geometry"] = fixed

        geometry_repairs.append({
            "city_name": city_name,
            "reason": reason,
            "area_difference_deg2": float(area_difference),
            "method": "shapely.make_valid",
        })

        print(
            f"[geometry repair, in memory only] {city_name}: {reason}",
            flush=True,
        )

    # --------------------------------------------------------
    # 5.2 Pre-check of inputs for all cities and all years
    # --------------------------------------------------------

    for city, (prefix, name, epsg) in CONFIG.items():
        boundary = boundaries[
            boundaries["name"].eq(name)
        ]

        require(
            len(boundary) == 1,
            f"SMOD cannot be matched uniquely: {city}",
        )

        urban = boundary.geometry.iloc[0]

        require(
            urban.is_valid and not urban.is_empty,
            f"SMOD geometry invalid: {city}",
        )

        grid_path = (
            BASE / "grids"
            / f"{city}_era5_grid.gpkg"
        )

        register(grid_path, "original_grid")

        grid = gpd.read_file(grid_path)

        require(
            grid.crs is not None
            and grid["cell_id"].is_unique,
            f"Abnormal grid: {city}",
        )

        selected = pts[
            pts["city"].eq(city)
        ].sort_values("cell_id")

        grid = (
            grid.to_crs(4326)
            .set_index("cell_id")
            .loc[selected["cell_id"]]
        )

        require(
            np.allclose(
                grid["lat_center"],
                selected["lat"],
                atol=1e-6,
                rtol=0,
            )
            and np.allclose(
                grid["lon_center"],
                selected["lon"],
                atol=1e-6,
                rtol=0,
            ),
            f"Grid-cell coordinates do not match: {city}",
        )

        cells = []

        for cell_id, row in grid.iterrows():
            require(
                row.geometry.is_valid
                and row.geometry.area > 0,
                f"Invalid study grid-cell geometry: {city} {cell_id}",
            )

            target = row.geometry.intersection(urban)

            areas = (
                gpd.GeoSeries(
                    [row.geometry, target],
                    crs=4326,
                )
                .to_crs(epsg)
                .area
                .to_numpy()
            )

            info = {
                "city": city,
                "cell_id": int(cell_id),
                "lat": float(row["lat_center"]),
                "lon": float(row["lon_center"]),
                "cell_area_m2": float(areas[0]),
                "smod_area_m2": float(areas[1]),
                "smod_cell_fraction": float(
                    areas[1] / areas[0]
                ),
            }

            grid_rows.append(info)

            cells.append((
                info,
                target,
                row.geometry.area,
            ))

        signature = None

        for year in YEARS:
            path, kind = source_for(city, year)

            register(
                path,
                "building_input",
                city=city,
                year=year,
                source_kind=kind,
            )

            with rasterio.open(path) as src:
                transform = src.transform

                require(
                    src.count >= 2
                    and src.crs is not None
                    and src.crs.to_epsg() == 4326,
                    f"Abnormal bands or CRS: {path}",
                )

                require(
                    transform.a > 0
                    and transform.e < 0
                    and transform.b == 0
                    and transform.d == 0,
                    f"Abnormal raster orientation: {path}",
                )

                require(
                    all(
                        src.scales[i] == 1
                        and src.offsets[i] == 0
                        for i in [0, 1]
                    ),
                    f"Abnormal band scaling: {path}",
                )

                current = (
                    tuple(transform),
                    src.width,
                    src.height,
                )

                if signature is None:
                    signature = current

                require(
                    current == signature,
                    f"{city} has different grids between years; check separately before computing.",
                )

            tasks.append((
                city,
                year,
                path,
                kind,
                cells,
            ))

        print(
            f"[input pre-check] {city}: all 8 yearly files present",
            flush=True,
        )

    # --------------------------------------------------------
    # 5.3 Create the new output folder
    # --------------------------------------------------------

    out = OUT_BASE

    out.mkdir(parents=True, exist_ok=True)
    (out / "by_city_year").mkdir(exist_ok=True)

    summary = {
        "status": "running",
        "cities": 35,
        "points": 652,
        "years": YEARS,
        "coverage_threshold": COVERAGE_MIN,

        "floor_height_m": 3.0,
        "smod_geometry_repairs": geometry_repairs,
    }

    def save_summary():
        (out / "summary.json").write_text(
            json.dumps(
                summary,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

    def save(df, filename):
        path = out / filename

        df.to_csv(
            path,
            index=False,
            encoding="utf-8-sig",
            float_format="%.17g",
        )

        back = pd.read_csv(
            path,
            encoding="utf-8-sig",
            float_precision="round_trip",
        )

        pd.testing.assert_frame_equal(
            df.reset_index(drop=True),
            back,
            check_dtype=False,
            check_exact=True,
        )

    save_summary()

    save(
        pd.DataFrame(sources),
        "input_manifest.csv",
    )

    save(
        pd.DataFrame(grid_rows),
        "study_grid_smod_areas.csv",
    )

    print("Output folder:", out, flush=True)

    # --------------------------------------------------------
    # 5.4 Compute and save city by city and year by year
    # --------------------------------------------------------

    all_rows = []

    try:
        for city, year, path, kind, cells in tasks:
            batch = []

            with rasterio.open(path) as src:
                for info, target, deg_area in cells:
                    record = dict(
                        info,
                        year=year,
                        source_kind=kind,
                        source_path=str(path),
                    )

                    record.update(
                        calculate_cell(
                            src,
                            target,
                            info["cell_area_m2"],
                            deg_area,
                        )
                    )

                    record["needs_review"] = (
                        not record["coverage_complete"]
                        or not record["height_available"]
                    )

                    batch.append(record)

            save(
                pd.DataFrame(batch),
                f"by_city_year/{city}_{year}.csv",
            )

            all_rows.extend(batch)

            summary.update(
                rows_finished=len(all_rows),
                city_years_finished=len(
                    list(
                        (out / "by_city_year").glob("*.csv")
                    )
                ),
            )

            save_summary()

            passed_count = sum(
                row["coverage_pass"]
                for row in batch
            )

            print(
                f"[{len(all_rows)}/5216] "
                f"{city} {year}: "
                f"coverage passed {passed_count}/{len(batch)}",
                flush=True,
            )

        # ----------------------------------------------------
        # 5.5 Verification
        # ----------------------------------------------------

        for record in sources:
            require(
                stamp(record["path"])
                == {
                    key: record[key]
                    for key in ["bytes", "mtime_ns"]
                },
                f"Input changed during computation: {record['path']}",
            )

        df = (
            pd.DataFrame(all_rows)
            .sort_values(KEY)
            .reset_index(drop=True)
        )

        require(
            len(df) == 5216
            and not df.duplicated(KEY).any(),
            "Abnormal number of outputs or keys.",
        )

        require(
            df.groupby(
                ["city", "cell_id"]
            )["study_pixels"].nunique().eq(1).all(),
            "Study-area pixel denominator varies between years.",
        )

        require(
            df.loc[
                ~df["coverage_pass"],
                METRICS,
            ].isna().all().all(),
            "Final metrics wrongly retained for low-coverage records.",
        )

        nonempty = df["eligible_height_model"]

        require(
            np.isfinite(
                df.loc[nonempty, METRICS].to_numpy(float)
            ).all(),
            "Non-finite metrics in qualifying building records.",
        )

        require(
            df["bld_coverage_observed"]
            .dropna()
            .between(0, 1)
            .all(),
            "Building coverage out of range.",
        )

        # ----------------------------------------------------
        # 5.6 Summary output
        # ----------------------------------------------------

        save(
            df,
            "building_smod_all_2016_2023.csv",
        )

        save(
            df[df["coverage_pass"]],
            "building_smod_coverage90.csv",
        )

        save(
            df[df["coverage_complete"]],
            "building_smod_complete100.csv",
        )

        save(
            df[df["needs_review"]],
            "building_rows_requiring_review.csv",
        )

        quality = (
            df.groupby(
                ["city", "year"],
                as_index=False,
            )
            .agg(
                rows=("cell_id", "size"),
                coverage90_rows=("coverage_pass", "sum"),
                complete100_rows=("coverage_complete", "sum"),
                height_model_rows=("eligible_height_model", "sum"),
                min_coverage=("coverage_fraction", "min"),
            )
        )

        save(
            quality,
            "city_year_quality.csv",
        )

        summary.update(
            status="calculation_complete",
            rows_finished=len(df),
            coverage90_rows=int(
                df["coverage_pass"].sum()
            ),
            complete100_rows=int(
                df["coverage_complete"].sum()
            ),
            height_model_rows=int(
                df["eligible_height_model"].sum()
            ),
            review_rows=int(
                df["needs_review"].sum()
            ),
        )

        save_summary()

    except (Exception, KeyboardInterrupt) as exc:
        summary.update(
            status="failed_or_interrupted",
            error=repr(exc),
            rows_finished=len(all_rows),
        )

        save_summary()
        raise

    print("Output folder:", out)

    return out, df


# ============================================================
# 6. Execution
# ============================================================

BUILDING_SMOD_DIR, BUILDING_SMOD_RESULTS = (
    run_building_smod()
)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

BUILDING_RUN = Path("../outputs/building")

FILLS = [
    ("Abidjan_IvoryCoast", 13, 2016, 2017),
]

KEY = ["city", "cell_id", "year"]


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def run_building_fill():
    src = BUILDING_RUN / "building_smod_all_2016_2023.csv"
    require(src.is_file(), f"Not found: {src}")
    df = pd.read_csv(src)
    require(not df.duplicated(KEY).any(), "Duplicate keys")
    formal = [c[: -len("_observed")] for c in df.columns
              if c.endswith("_observed") and c[: -len("_observed")] in df.columns]
    require(len(formal) > 0, "No final building columns found")

    before = df.copy()
    df["bld_filled"] = False
    df["bld_fill_source_year"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
    report = {"source": str(src), "formal_columns": formal, "fills": []}

    for city, cell, year, donor_year in FILLS:
        tgt = df.index[(df.city == city) & (df.cell_id == cell) & (df.year == year)]
        don = df.index[(df.city == city) & (df.cell_id == cell) & (df.year == donor_year)]
        require(len(tgt) == 1 and len(don) == 1, f"{city} {cell}: row for {year} or {donor_year} not found")
        t, d = tgt[0], don[0]
        require(not bool(df.at[t, "coverage_pass"]), f"{city} {cell} {year} already meets the coverage threshold; no filling needed")
        require(bool(df.at[d, "coverage_pass"]), f"{city} {cell} {donor_year} does not meet the coverage threshold; cannot be used as the donor")
        require(df.loc[t, formal].isna().all(), f"{city} {cell} {year} final columns should be NaN")
        require(df.loc[d, formal].notna().all(), f"{city} {cell} {donor_year} final columns contain NaN")

        df.loc[t, formal] = df.loc[d, formal].to_numpy()
        df.at[t, "eligible_density_model"] = True
        df.at[t, "eligible_height_model"] = True
        df.at[t, "bld_filled"] = True
        df.at[t, "bld_fill_source_year"] = donor_year
        report["fills"].append(dict(
            city=city, cell_id=int(cell), year=int(year), source_year=int(donor_year),
            coverage_fraction=float(df.at[t, "coverage_fraction"]),
            source_coverage_fraction=float(df.at[d, "coverage_fraction"]),
            observed_vs_filled={c: [None if pd.isna(df.at[t, c + "_observed"]) else float(df.at[t, c + "_observed"]),
                                    float(df.at[t, c])]
                                for c in ["bld_coverage", "bld_mean_h_m", "bld_std_h_m", "bld_FAR"]},
        ))

    changed = df["bld_filled"]
    keep_cols = [c for c in before.columns if c not in formal + ["eligible_density_model", "eligible_height_model"]]
    pd.testing.assert_frame_equal(before[keep_cols], df[keep_cols], check_dtype=False)
    pd.testing.assert_frame_equal(before.loc[~changed, formal], df.loc[~changed, formal], check_dtype=False)

    out = BUILDING_RUN / "building_smod_all_2016_2023_filled.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig", float_format="%.17g")
    back = pd.read_csv(out, float_precision="round_trip")
    back["bld_fill_source_year"] = back["bld_fill_source_year"].astype("Int64")
    pd.testing.assert_frame_equal(df, back, check_dtype=False, check_exact=True)
    (BUILDING_RUN / "building_fill_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Output:", out)
    return df


BUILDING_FILLED = run_building_fill()